# Data Ingestion

### Install Kaggle API and required libraries

In [0]:
%pip install kaggle
import os
from pyspark.sql.functions import current_timestamp
import subprocess

### Set up Kaggle credentials

1. Go to kaggle.com/settings/API Tokens
2. Click "Create New Token" and follow the instructions
3. Set credentials below

In [0]:
os.environ['KAGGLE_USERNAME'] = '' # Kaggle username
os.environ['KAGGLE_KEY'] = '' # API key


### Create a Unity Catalog Volume for large file storage

In [0]:
spark.sql("CREATE CATALOG IF NOT EXISTS scientific_trend_analysis")
spark.sql("CREATE SCHEMA IF NOT EXISTS scientific_trend_analysis.lds")
spark.sql("CREATE VOLUME IF NOT EXISTS scientific_trend_analysis.lds.arxiv_raw")


### Download the arXiv metadata dataset to Unity Catalog Volume

In [0]:
volume_path = "/Volumes/scientific_trend_analysis/lds/arxiv_raw"
file_path = f"{volume_path}/arxiv-metadata-oai-snapshot.json"

if os.path.exists(file_path):
    print(f"File already exists at {file_path}, skipping download")
else:
    subprocess.run([
        'kaggle', 'datasets', 'download', '-d', 'Cornell-University/arxiv',
        '-p', volume_path, '--unzip'
    ], check=True)

### Load the JSON file with Spark

In [0]:
df_raw = spark.read.json(file_path)
df_bronze = df_raw.withColumn("ingest_time", current_timestamp())

print(f"Loaded {df_bronze.count():,} records")

### Save to Unity Catalog table

In [0]:
df_bronze.write.format("delta").mode("overwrite").saveAsTable("scientific_trend_analysis.lds.arxiv_bronze")

print("Data successfully loaded")
display(df_bronze.limit(5))